
# StereoQueerEval — Task B: Hate Speech Detection (mmBERT only)

Phiên bản rút gọn chỉ huấn luyện **Task B (hate speech detection)** với **mmBERT frozen/semi-frozen**.
Bỏ hẳn chế độ scratch (RNN/LSTM/Transformer tự học embedding) và 2 subtask còn lại (stereotype, target).

### Task B: 3 lớp `no` / `yes_implicit` / `yes_explicit`

Dùng **2 head nhị phân** rồi ghép lại thành phân phối 3 lớp theo **cây phân rã xác suất**:

```
p_hate     = sigmoid(hate_head)      # P(có hate) = P(yes_implicit) + P(yes_explicit)
p_implicit = sigmoid(subtype_head)   # P(yes_implicit | có hate)

P(No)       = 1 - p_hate
P(Implicit) = p_hate * p_implicit
P(Explicit) = p_hate * (1 - p_implicit)
```

Dự đoán = `argmax` trên 3 giá trị trên. Không hard-gate ở bước 1, nên không bị
error propagation, và đầu ra vẫn là flat 3-class đúng cách chấm của SemEval.

| Head | Nhãn | Loại | Loss | Đánh giá riêng |
|---|---|---|---|---|
| `hate_head` | `no` vs `yes_*` | nhị phân | BCEWithLogits | accuracy + macro-F1 nhị phân |
| `subtype_head` | `yes_implicit` vs `yes_explicit` | nhị phân | BCEWithLogits, **chỉ trên mẫu có hate** | F1 từng lớp |

Đánh giá tổng: **macro-F1 3 lớp** (chỉ số chính của SemEval) + confusion matrix.

### Vì sao cây phân rã thay vì 1 head 3 lớp phẳng?
- Head `subtype` chỉ học phân biệt `implicit` vs `explicit`, **không bị lớp `no` chiếm 48.4%
  dữ liệu phân loãng** — đây là chỗ macro-F1 hay bị mất nhất ở mô hình phẳng.
- Vẫn giữ được một đầu `yes`/`no` độc lập để xem xác suất có hate hay không.
- Tổng `P(Implicit) + P(Explicit) = p_hate` nên hai head vẫn nhất quán, không mâu thuẫn.

### Điểm khác biệt so với bản Task A
1. **Chỉ train task B** — bỏ `stereotype` (Task A) và `target`
2. **Cây 2 head nhị phân** thay cho head 1-logit BCE của Task A
3. **Output có suffix `_taskB`** — không ghi đè kết quả Task A

> Lưu ý: `stereotype=yes` (Task A) ≠ `hate_speech=yes` (Task B). 1582 mẫu hate=yes nhưng
> stereotype=no, và 491 mẫu stereotype=yes nhưng hate=no. Task B chỉ dùng cột `hate_speech`.

---


In [ ]:
!pip uninstall -y transformers
!pip install "transformers==4.56.2"


## Dữ liệu
- 3 file TSV UTF-8: `StereoQueerEval_{EN,IT,NL}_training.tsv`.
- **Cảnh báo:** file có ký tự xuống dòng `\n` **bên trong field** → bắt buộc dùng
  `pandas.read_csv(sep='\t')`, không đọc bằng công cụ theo dòng.

| File | Số comment |
|---|---|
| EN | 2.989 |
| IT | 2.400 |
| NL | 2.238 |
---


In [ ]:

import os
import re
import glob
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Notebook nằm trong /notebook, dữ liệu nằm ở ../LGBT
candidates = [
    '../LGBT/*_training.tsv',
    'LGBT/*_training.tsv',
    '../**/StereoQueerEval_*_training.tsv',
]
paths = sorted({p for pat in candidates for p in glob.glob(pat, recursive=True)})
assert paths, 'Không tìm thấy file dữ liệu StereoQueerEval (*_training.tsv)'
print(paths)



## Cấu hình: chỉ dùng mmBERT với Transformer head


In [ ]:
import random

SEED = 42   # seed cho training, độc lập với best_seed dùng để chia tập

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

In [ ]:
MMBERT_MODEL_NAME = 'EuroBERT/EuroBERT-210m'
MMBERT_DIM = 768

MMBERT_UNFREEZE_LAYERS = 2

# --- LR mới cho từng pha (thay cho lr=1e-6 cố định cũ) ---
PHASE1_HEAD_LR = 3e-4     # pha 1: head/pooling train từ đầu
PHASE2_HEAD_LR = 5e-5     # pha 2: head fine-tune nhẹ
PHASE2_BACKBONE_LR = 5e-6 # pha 2: backbone mở, LR nhỏ hơn head
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

TWO_PHASE = True
FREEZE_PHASE_EPOCHS = 5
UNFREEZE_PHASE_EPOCHS = 15


## Loss & device setup


In [ ]:

# Task B: 2 head nhị phân, ghép thành phân phối 3 lớp theo cây phân rã
loss_hate = nn.BCEWithLogitsLoss()      # P(no) vs P(yes_implicit) + P(yes_explicit)
loss_subtype = nn.BCEWithLogitsLoss()   # P(yes_implicit | hate) vs P(yes_explicit | hate)

SUBTYPE_WEIGHT = 1.0   # đặt 0 -> bỏ head subtype, chỉ còn nhị phân (không dùng khi nộp Task B)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang huấn luyện trên: {device}")


In [ ]:

dfs = []
for p in paths:
    lang = re.search(r'_([A-Z]{2})_training\.tsv$', p).group(1)
    df = pd.read_csv(p, sep='\t', quoting=1)   # quan trọng: field có xuống dòng
    df['lang'] = lang
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
print(f"Tổng số comment: {len(df_all)}")
print(df_all.groupby('lang').size())


In [ ]:

def safe_clean(text):
    text = str(text).lower()
    # Chễ gộp xuống dòng, KHÔNG xoá ký tự đặc biệt/có dấu (quan trọng với IT & NL)
    text = re.sub(r'[\n\t\r]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_all['text'] = (
    'someone commented: ' + df_all['yt_comment'] + ' from the video named: ' + df_all['yt_title']
    # + ' [SEP] ' + df_all['yt_description']
).map(safe_clean)

print(df_all[['lang', 'text']].head(3).to_string())


In [ ]:

# Nhãn Task B: flat 3 lớc + 2 nhị phân con theo cây phân rã
HS_LABELS = ['no', 'yes_implicit', 'yes_explicit']

unknown = set(df_all['hate_speech'].unique()) - set(HS_LABELS)
assert not unknown, f'Nhãn hate_speech lạ: {unknown}'

# flat 3 lớc: dùng để chia tầng + làm ground truth khi đánh giá
df_all['hs_y'] = df_all['hate_speech'].map({l: i for i, l in enumerate(HS_LABELS)}).astype(np.int64)

# head 1: 0 = no, 1 = yes_implicit hoặc yes_explicit
df_all['hate_bin'] = (df_all['hate_speech'] != 'no').astype(np.float32).values

# head 2: 1 = yes_implicit, 0 = yes_explicit, -1 = mẫu 'no' (bị mask khỏi loss vì
# P(implicit | hate) không định nghĩa khi không có hate)
df_all['subtype_bin'] = np.where(
    df_all['hate_speech'] == 'yes_implicit', 1.0,
    np.where(df_all['hate_speech'] == 'yes_explicit', 0.0, -1.0)).astype(np.float32)

print('--- hate_speech ---')
print(df_all['hate_speech'].value_counts())

print('\n--- ngôn ngữ x hate_speech ---')
print(pd.crosstab(df_all['lang'], df_all['hate_speech']))

print('\n--- head 1: hate_bin ---')
print(df_all['hate_bin'].value_counts().sort_index())

print('\n--- head 2: subtype_bin (chỉ mẫu có hate được huấn luyện) ---')
print(df_all['subtype_bin'].value_counts().sort_index())

print('\n--- stereotype (Task A) chỉ để tham khảo, KHÔNG dùng cho Task B ---')
print(pd.crosstab(df_all['stereotype'], df_all['hate_speech']))



## Chia dữ liệu theo video — chống rò rỉ
1 video (1 `yt_title`) chỉ xuất hiện ở 1 tập, vì title/description là **input**:
nếu cùng video nằm ở cả train lẫn val, model sẽ "nhìn thấy" context trước.


In [ ]:

import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

# Khóa phân tầng = ngôn ngữ + nhãn Task B (vd: 'IT_2'), để cân cả hai cùng lúc
strat_key = (df_all['lang'].astype(str) + '_' + df_all['hs_y'].astype(str)).values
groups    = df_all['yt_title'].values

# Tỉ lệ chuẩn của từng ô (lang, label) trên toàn bộ dữ liệu
all_ratio = df_all.assign(k=strat_key).groupby('k').size() / len(df_all)

def split_score(val_idx):
    """Độ lệch tối đa giữa tỉ lệ ô (lang,label) trong val so với toàn bộ + lệch kích thước so với 20%."""
    v = pd.Series(strat_key[val_idx]).value_counts() / len(val_idx)
    v = v.reindex(all_ratio.index).fillna(0)
    ratio_dev = (v - all_ratio).abs().max()
    size_dev  = abs(len(val_idx) / len(df_all) - 0.20)
    return ratio_dev + size_dev

best = None
for seed in range(1000):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    tr_idx, va_idx = next(iter(sgkf.split(df_all, strat_key, groups=groups)))  # fold đầu = val 20%
    s = split_score(va_idx)
    if best is None or s < best[0]:
        best = (s, seed, tr_idx, va_idx)

score, best_seed, train_idx, val_idx = best
df_train = df_all.iloc[train_idx].reset_index(drop=True)
df_val   = df_all.iloc[val_idx].reset_index(drop=True)

# --- Kiểm tra ---
overlap = len(set(df_train['yt_title']) & set(df_val['yt_title']))
print(f"Seed chọn: {best_seed} | score lệch: {score:.4f}")
print(f"Train: {len(df_train)} ({len(df_train)/len(df_all):.1%}) | "
      f"Val: {len(df_val)} ({len(df_val)/len(df_all):.1%}) | video trùng lặp: {overlap}")

print("\nTỉ lệ từng nhãn Task B theo ngôn ngữ:")
cmp = pd.DataFrame({
    'all_n':   df_all.groupby('lang').size(),
    'train_n': df_train.groupby('lang').size(),
    'val_n':   df_val.groupby('lang').size(),
    'all_no':       df_all.groupby('lang')['hs_y'].apply(lambda s: (s == 0).mean()).round(3),
    'all_implicit': df_all.groupby('lang')['hs_y'].apply(lambda s: (s == 1).mean()).round(3),
    'all_explicit': df_all.groupby('lang')['hs_y'].apply(lambda s: (s == 2).mean()).round(3),
    'train_no':      df_train.groupby('lang')['hs_y'].apply(lambda s: (s == 0).mean()).round(3),
    'val_no':        df_val.groupby('lang')['hs_y'].apply(lambda s: (s == 0).mean()).round(3),
})
print(cmp)

print("\nTỉ lệ ngôn ngữ trong từng tập:")
print(pd.DataFrame({
    'all':   df_all['lang'].value_counts(normalize=True),
    'train': df_train['lang'].value_counts(normalize=True),
    'val':   df_val['lang'].value_counts(normalize=True),
}).round(3))



## mmBERT setup & helper functions


In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
from transformers import AutoConfig
from transformers.modeling_rope_utils import ROPE_INIT_FUNCTIONS

config = AutoConfig.from_pretrained(
    MMBERT_MODEL_NAME,
    trust_remote_code=True
)

print("Transformers:", transformers.__version__)
print("model_type:", config.model_type)
print("rope_type:", getattr(config, "rope_type", None))
print("ROPE:", ROPE_INIT_FUNCTIONS.keys())

In [ ]:
import re


def layer_index_in_name(name):
    """Lấy chỉ số tầng từ TÊN THAM SỐ (không cần biết cấu trúc module):
    - mmBERT/ModernBERT: 'layers.21.attn.Wqkv.weight'      -> 21
    - BERT cổ           : 'encoder.layer.9.attention...'   -> 9
    - không phải tầng   : 'embeddings.tok_embeddings'      -> None
    """
    m = re.search(r'(?:^|\.)(?:layers|layer|blocks)\.(\d+)\b', name)
    return int(m.group(1)) if m else None


def count_encoder_blocks(model):
    """Số tầng encoder, đếm theo chỉ số block trong TÊN THAM SỐ (không cần biết
    cấu trúc module của transformers — phiên bản nào cũng chạy)."""
    idx = {layer_index_in_name(n) for n, _ in model.named_parameters()}
    idx.discard(None)
    return len(idx)


def unfreeze_last_n(model, n):
    """Mở requires_grad cho n tầng encoder CUỐI theo chỉ số block trong tên tham số
    (ModernBERT: 'encoder.layers.X.*'; BERT cổ: 'encoder.layer.X.*'). Trả số tầng tìm thấy."""
    groups = {}
    for name, p in model.named_parameters():
        i = layer_index_in_name(name)
        if i is not None:
            groups.setdefault(i, []).append(p)
    if not groups:
        return 0
    for i in sorted(groups)[-n:]:
        for p in groups[i]:
            p.requires_grad = True
    return len(groups)

In [ ]:

from transformers import AutoTokenizer, AutoModel

tok = AutoTokenizer.from_pretrained(MMBERT_MODEL_NAME, trust_remote_code=True)

# In-model: backbone giữ trên GPU suốt lúc train; freeze trước rồi mở N tầng cuối.
mmbert_backbone = AutoModel.from_pretrained(MMBERT_MODEL_NAME, trust_remote_code=True).eval().to(device)
for p in mmbert_backbone.parameters():
    p.requires_grad = False

if MMBERT_UNFREEZE_LAYERS > 0 and not TWO_PHASE:
    n_blocks = unfreeze_last_n(mmbert_backbone, MMBERT_UNFREEZE_LAYERS)
    if n_blocks:
        print(f'mmBERT: {n_blocks} tầng -> fine-tune '
              f'{min(MMBERT_UNFREEZE_LAYERS, n_blocks)} tầng cuối (LR {PHASE2_BACKBONE_LR})')
    else:
        sample = ' | '.join(n for n, _ in list(mmbert_backbone.named_parameters())[:6])
        print('⚠️ Không tìm thấy tầng encoder trong tên tham số. Mẫu tên:', sample)
elif TWO_PHASE:
    print('Phương án 2-pha: mmBERT đông cứng ở pha 1, sẽ mở '
          f'{MMBERT_UNFREEZE_LAYERS} tầng cuối ở pha 2')
else:
    print('mmBERT: đông cứng toàn bộ (feature extractor)')

print('MMBertTransformerModel: tokenize trong dataset')



## MMBertSeqDataset (Task B: 2 nhãn nhị phân)


In [ ]:

class MMBertSeqDataset(Dataset):
    """Tokenize bằng AutoTokenizer của mmBERT một lần lúc khởi tạo rồi cache.
    Mang 2 nhãn nhị phân của Task B: `hate_bin` (no vs yes_*) và
    `subtype_bin` (yes_implicit vs yes_explicit, -1 = mẫu 'no' bị mask).
    """

    def __init__(self, texts, hate_labels, subtype_labels, tok, max_len=256):
        self.hate_labels = np.asarray(hate_labels, dtype=np.float32)
        self.subtype_labels = np.asarray(subtype_labels, dtype=np.float32)
        self.ids, self.mask = self._encode(list(texts), tok, max_len)

    @staticmethod
    def _encode(texts, tok, max_len):
        # EuroBERT không sử dụng BERT-style [SEP].
        # Không thay [SEP] bằng tok.sep_token vì tok.sep_token == None.
        texts = [str(t) for t in texts]

        enc = tok(
            texts,
            truncation=True,
            max_length=max_len,
            padding='max_length',
            return_tensors='pt'
        )

        return enc['input_ids'], enc['attention_mask']

    def __len__(self):
        return len(self.hate_labels)

    def __getitem__(self, idx):
        return (
            self.ids[idx],
            self.mask[idx],
            torch.tensor(self.hate_labels[idx], dtype=torch.float32),
            torch.tensor(self.subtype_labels[idx], dtype=torch.float32)
        )


In [ ]:
class MMBertTransformerModel(nn.Module):
    """mmBERT (đông cứng) + TransformerEncoder tự dựng trên (B, L, 768) + stereotype head."""
    def __init__(self, mmbert_model, d_model=768, num_heads=8, num_layers=4,
                 dim_feedforward=1024, dropout=0.3):
        super(MMBertTransformerModel, self).__init__()
        self.mmbert = mmbert_model   # backbone (mở/đông cứng do requires_grad, đổi được giữa 2 pha)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads,
            dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Chỉ có stereotype head
        self.st_head = nn.Sequential(nn.Linear(d_model, d_model // 2), nn.ReLU(),
                                     nn.Dropout(dropout), nn.Linear(d_model // 2, 1))

    def forward(self, input_ids, attention_mask):
        # 1. mmBERT: chỉ lấy feature từng token, KHÔNG pool.
        backbone_trainable = any(p.requires_grad for p in self.mmbert.parameters())
        if backbone_trainable:
            seq_hidden = self.mmbert(input_ids=input_ids,
                                     attention_mask=attention_mask).last_hidden_state
        else:
            with torch.no_grad():
                seq_hidden = self.mmbert(input_ids=input_ids,
                                         attention_mask=attention_mask).last_hidden_state
        seq_hidden = seq_hidden.to(torch.float32)          # (B, L, 768)

        # 2. Transformer custom với padding mask (True = vị trí PAD)
        x = self.transformer_encoder(seq_hidden,
                                     src_key_padding_mask=(attention_mask == 0))

        # 3. Masked mean pooling (không cộng pad vào)
        m = attention_mask.unsqueeze(-1).float()
        x = (x * m).sum(dim=1) / m.sum(dim=1).clamp(min=1e-9)

        # Chỉ trả về st_logits
        return self.st_head(x)

## Backbone + MixPooling + Head

In [ ]:

class AttentionPooling(nn.Module):
    """1 learnable query attend vào các token quan trọng thay vì mean đều."""
    def __init__(self, d_model=768):
        super().__init__()
        self.query = nn.Parameter(torch.randn(d_model) * 0.02)
        self.scale = d_model ** -0.5

    def forward(self, hidden, attention_mask):
        scores = (hidden @ self.query) * self.scale
        scores = scores.masked_fill(attention_mask == 0, float('-inf'))
        weights = torch.softmax(scores, dim=-1).unsqueeze(-1)
        return (hidden * weights).sum(dim=1)


class HybridPooling(nn.Module):
    """Ghép mean + max + attention pooling -> (B, 3*D)."""
    def __init__(self, d_model=768):
        super().__init__()
        self.attn = AttentionPooling(d_model)

    def forward(self, hidden, attention_mask):
        m = attention_mask.unsqueeze(-1).float()                        # (B, L, 1)
        mean = (hidden * m).sum(dim=1) / m.sum(dim=1).clamp(min=1e-9)
        mx = hidden.masked_fill(m == 0, float('-inf')).max(dim=1).values
        att = self.attn(hidden, attention_mask)
        return torch.cat([mean, mx, att], dim=-1)


class MMBertHybridPoolModel(nn.Module):
    """mmBERT (đông cứng/mở) + Hybrid Pooling + 2 head nhị phân cho Task B.

    forward trả về (hate_logit, subtype_logit):
      - hate_logit   -> sigmoid = p_hate     = P(yes_implicit) + P(yes_explicit)
      - subtype_logit-> sigmoid = p_implicit = P(yes_implicit | hate)

    Ba xác suất lớp suy ra bằng `compound_probs` (xem cell Evaluation).
    """

    def __init__(self, mmbert_model, d_model=768, dropout=0.2, input_dropout=0.3):
        super().__init__()
        self.mmbert = mmbert_model
        self.pool = HybridPooling(d_model)
        in_dim = d_model * 3
        self.head_norm = nn.LayerNorm(in_dim)

        def mlp(out_dim):
            return nn.Sequential(
                nn.Dropout(input_dropout),
                nn.Linear(in_dim, d_model // 2), nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(d_model // 2, out_dim))

        self.hate_head = mlp(1)      # (B,) logit P(có hate)
        self.subtype_head = mlp(1)   # (B,) logit P(implicit | hate)

    def forward(self, input_ids, attention_mask):
        backbone_trainable = any(p.requires_grad for p in self.mmbert.parameters())
        if backbone_trainable:
            seq_hidden = self.mmbert(input_ids=input_ids,
                                      attention_mask=attention_mask).last_hidden_state
        else:
            with torch.no_grad():
                seq_hidden = self.mmbert(input_ids=input_ids,
                                          attention_mask=attention_mask).last_hidden_state
        seq_hidden = seq_hidden.to(torch.float32)
        pooled = self.pool(seq_hidden, attention_mask)      # (B, 3D)
        z = self.head_norm(pooled)
        return self.hate_head(z).squeeze(-1), self.subtype_head(z).squeeze(-1)



## Evaluation functions — ghép 2 head theo cây phân rã (Task B)


In [ ]:

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix


def compound_probs(hate_logit, subtype_logit):
    """Cây phân rã 3 lớp từ 2 head nhị phân:
        P(No)       = 1 - p_hate
        P(Implicit) = p_hate * p_implicit
        P(Explicit) = p_hate * (1 - p_implicit)
    Trả về tensor (B, 3), tổng mỗi hàng = 1."""
    p_hate = torch.sigmoid(hate_logit)
    p_imp = torch.sigmoid(subtype_logit)
    return torch.stack([1.0 - p_hate, p_hate * p_imp, p_hate * (1.0 - p_imp)], dim=-1)


@torch.no_grad()
def predict(model, loader):
    """Trả về (nhãn 3 lớp, ma trận xác suất 3 lớp, xác suất p_hate)."""
    model.eval()
    pred_l, prob_l, hate_l = [], [], []
    for ids, mask, _, _ in loader:
        hate_logit, sub_logit = model(ids.to(device), mask.to(device))
        prob = compound_probs(hate_logit, sub_logit)
        pred_l.append(prob.argmax(dim=-1).cpu().numpy())
        prob_l.append(prob.cpu().numpy())
        hate_l.append(torch.sigmoid(hate_logit).cpu().numpy())
    return np.concatenate(pred_l), np.concatenate(prob_l), np.concatenate(hate_l)


def evaluate(model, df, loader):
    pred, prob, p_hate = predict(model, loader)
    y = df['hs_y'].values

    metrics = {
        # --- chỉ số chính của Task B: flat 3 lớp ---
        'hs_f1_macro': f1_score(y, pred, average='macro', zero_division=0),
        'hs_acc': accuracy_score(y, pred),
        'f1_no': f1_score(y, pred, labels=[0], average='macro', zero_division=0),
        'f1_yes_implicit': f1_score(y, pred, labels=[1], average='macro', zero_division=0),
        'f1_yes_explicit': f1_score(y, pred, labels=[2], average='macro', zero_division=0),
        # --- head 1 đánh giá riêng như một bài nhị phân no/yes ---
        'hate_acc': accuracy_score(df['hate_bin'].values, (p_hate >= 0.5).astype(int)),
        'hate_f1': f1_score(df['hate_bin'].values, (p_hate >= 0.5).astype(int),
                            average='macro', zero_division=0),
    }
    return metrics, pred, prob, p_hate


def print_metrics(metrics, name='VALIDATION'):
    print(f'--- {name} ---')
    for k, v in metrics.items():
        print(f'  {k}: {v:.4f}')



## Training loop (Task B)


In [ ]:
from transformers import get_linear_schedule_with_warmup

def build_optimizer_and_scheduler(model, head_lr, backbone_lr, num_training_steps,
                                   warmup_ratio=WARMUP_RATIO, weight_decay=WEIGHT_DECAY):
    no_decay = ['bias', 'LayerNorm.weight', 'layer_norm.weight', 'norm.weight', 'query']

    def split_decay(params):
        decay, nodecay = [], []
        for n, p in params:
            (nodecay if any(nd in n for nd in no_decay) else decay).append(p)
        return decay, nodecay

    mmbert_params = [(n, p) for n, p in model.named_parameters()
                      if p.requires_grad and n.startswith('mmbert.')]
    head_params   = [(n, p) for n, p in model.named_parameters()
                      if p.requires_grad and not n.startswith('mmbert.')]

    groups = []
    if mmbert_params:
        d, nd = split_decay(mmbert_params)
        groups += [{'params': d, 'lr': backbone_lr, 'weight_decay': weight_decay},
                   {'params': nd, 'lr': backbone_lr, 'weight_decay': 0.0}]
    d, nd = split_decay(head_params)
    groups += [{'params': d, 'lr': head_lr, 'weight_decay': weight_decay},
               {'params': nd, 'lr': head_lr, 'weight_decay': 0.0}]

    optimizer = torch.optim.AdamW(groups)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(warmup_ratio * num_training_steps),
        num_training_steps=num_training_steps)
    return optimizer, scheduler

In [ ]:

def train_model(model, tag, optimizer, scheduler, num_epochs=20, patience=5):
    best_val_loss = float('inf')
    best_val_f1 = -1.0
    counter = 0
    early_stop = False

    print("Starting the training process with Early Stopping...")
    for epoch in range(num_epochs):
        if early_stop:
            print("🛑 Early stopping triggered. Training finished!")
            break

        start_time = time.time()
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            ids, mask, hate_bin, sub_bin = batch
            ids, mask = ids.to(device), mask.to(device)
            hate_bin, sub_bin = hate_bin.to(device), sub_bin.to(device)

            hate_logit, sub_logit = model(ids, mask)
            loss = loss_hate(hate_logit, hate_bin)

            # head subtype chỉ học trên mẫu CÓ hate (sub_bin >= 0), vì
            # P(implicit | hate) không định nghĩa với mẫu 'no' (sub_bin == -1)
            if SUBTYPE_WEIGHT > 0:
                m = sub_bin >= 0
                if m.any():
                    loss = loss + SUBTYPE_WEIGHT * loss_subtype(sub_logit[m], sub_bin[m])

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                ids, mask, hate_bin, sub_bin = batch
                ids, mask = ids.to(device), mask.to(device)
                hate_bin, sub_bin = hate_bin.to(device), sub_bin.to(device)

                hate_logit, sub_logit = model(ids, mask)
                batch_loss = loss_hate(hate_logit, hate_bin)
                if SUBTYPE_WEIGHT > 0:
                    m = sub_bin >= 0
                    if m.any():
                        batch_loss = batch_loss + SUBTYPE_WEIGHT * loss_subtype(sub_logit[m], sub_bin[m])
                val_loss += batch_loss.item()

        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)
        metrics, _, _, _ = evaluate(model, df_val, val_loader)
        val_f1 = metrics['hs_f1_macro']

        print(f"Epoch {epoch+1} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f} "
              f"| Time: {time.time()-start_time:.1f}s")
        print_metrics(metrics, f'epoch {epoch+1}')

        # --- Lưu bản tốt nhất theo macro-F1 (dùng để đánh giá/deploy cuối) ---
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), f'{tag}_bestf1.pt')
            print(f"  💾 macro-F1 mới tốt nhất: {val_f1:.4f} -> lưu {tag}_bestf1.pt")

        # --- Early stopping vẫn dựa theo val_loss ---
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), f'{tag}_bestloss.pt')
            counter = 0
        else:
            counter += 1
            print(f"⚠️ No improvement. Patience counter: {counter}/{patience}")
            if counter >= patience:
                early_stop = True

    return best_val_f1, best_val_loss


In [ ]:
print("sep_token     =", repr(tok.sep_token))
print("sep_token_id  =", tok.sep_token_id)
print("all_special_tokens =", tok.all_special_tokens)

n_sep = df_train['text'].astype(str).str.contains(
    r'\[SEP\]',
    regex=True
).sum()

print("Samples containing [SEP]:", n_sep)


## Dataset & Model initialization


In [ ]:

BATCH_SIZE = 64
MAX_LEN = 256

set_seed(SEED)

# Dataset mang 2 nhãn nhị phân (hate_bin, subtype_bin)
train_dataset = MMBertSeqDataset(df_train['text'].values, df_train['hate_bin'].values,
                                 df_train['subtype_bin'].values, tok, MAX_LEN)
val_dataset   = MMBertSeqDataset(df_val['text'].values, df_val['hate_bin'].values,
                                 df_val['subtype_bin'].values, tok, MAX_LEN)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader    = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

final_tag = 'best_mmbert_tf_taskB'
# model = MMBertTransformerModel(mmbert_backbone, d_model=MMBERT_DIM).to(device)

model = MMBertHybridPoolModel(mmbert_backbone, d_model=MMBERT_DIM, dropout=0.2, input_dropout=0.2).to(device)


In [ ]:

# # 1. In toàn bộ kiến trúc model
# print(model)
# print("=" * 80)

# # 2. Đếm tổng số tham số + số tham số trainable
# total_params = sum(p.numel() for p in model.parameters())
# trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# print(f"Tổng số tham số      : {total_params:,}")
# print(f"Số tham số trainable : {trainable_params:,}")
# print(f"Số tham số frozen    : {total_params - trainable_params:,}")
# print("=" * 80)

# # 3. Liệt kê chi tiết từng layer/module con, số tham số và trạng thái freeze/train
# def print_layer_params(model):
#     print(f"{'Module':<55}{'Params':>15}{'Trainable':>12}")
#     print("-" * 82)
#     for name, module in model.named_modules():
#         # chỉ in các module lá (không có sub-module con) để tránh trùng lặp
#         if len(list(module.children())) == 0 and name != '':
#             n_params = sum(p.numel() for p in module.parameters())
#             if n_params == 0:
#                 continue
#             is_trainable = any(p.requires_grad for p in module.parameters())
#             print(f"{name:<55}{n_params:>15,}{str(is_trainable):>12}")

# print_layer_params(model)
# print("=" * 80)

# # 4. Gộp theo nhóm lớn: mmbert backbone (theo từng encoder layer) / pooling / head
def print_grouped_params(model):
    groups = {}
    for name, p in model.named_parameters():
        if name.startswith('mmbert.'):
            idx = layer_index_in_name(name)   # đã định nghĩa sẵn, xử lý cả 'layers.N' lẫn 'layer.N'
            key = f'mmbert.layers.{idx}' if idx is not None else 'mmbert (embeddings/other)'
        elif name.startswith('pool.'):
            key = 'attention_pooling'
        elif name.startswith('hate_head.'):
            key = 'hate_head (gate)'
        elif name.startswith('subtype_head.'):
            key = 'subtype_head (implicit/explicit)'
        else:
            key = name.split('.')[0]
        groups.setdefault(key, {'params': 0, 'trainable': False})
        groups[key]['params'] += p.numel()
        if p.requires_grad:
            groups[key]['trainable'] = True

    def sort_key(k):
        # sắp xếp: các layer theo số thứ tự, còn lại đẩy xuống cuối
        if k.startswith('mmbert.layers.'):
            return (0, int(k.split('.')[-1]))
        return (1, k)

    print(f"{'Group':<35}{'Params':>15}{'Trainable':>12}")
    print("-" * 62)
    for k in sorted(groups, key=sort_key):
        v = groups[k]
        print(f"{k:<35}{v['params']:>15,}{str(v['trainable']):>12}")

# print_grouped_params(model)


In [ ]:
def freeze_mmbert(n_unfreeze):
    """Đông cứng toàn bộ mmBERT, sau đó mở n_unfreeze tầng encoder cuối
    (thao tác theo chỉ số block trong TÊN THAM SỐ — version-proof)."""
    for p in mmbert_backbone.parameters():
        p.requires_grad = False
    if n_unfreeze > 0:
        n_blocks = unfreeze_last_n(mmbert_backbone, n_unfreeze)
        if n_blocks:
            opened = min(n_unfreeze, n_blocks)
            n_trainable = sum(1 for p in mmbert_backbone.parameters() if p.requires_grad)
            print(f'> mmBERT: {n_blocks} tầng -> mở {opened} tầng cuối '
                  f'({n_trainable} params trainable)')
        else:
            sample = ' | '.join(n for n, _ in list(mmbert_backbone.named_parameters())[:6])
            print('⚠️ Không tìm thấy tầng encoder trong tên tham số. Mẫu tên:', sample)
    else:
        print('> mmBERT đông cứng toàn bộ')

In [ ]:
freeze_mmbert(MMBERT_UNFREEZE_LAYERS)   # gọi trước
print_grouped_params(model)             # rồi mới in

In [ ]:

print(sum(p.numel() for n, p in model.named_parameters() if not n.startswith('mmbert.')))
# kỳ vọng ~1.05M tham số phần head (2 MLP 1-logit, input 3*768 nên nhiều hơn bản Task A ~890K)

# xác nhận hidden_states[-1] có trùng last_hidden_state không (chỉ để biết thang giá trị)
with torch.no_grad():
    b = next(iter(train_loader))
    ids, mask = b[0][:2].to(device), b[1][:2].to(device)
    o = model.mmbert(input_ids=ids, attention_mask=mask, output_hidden_states=True)
    print(torch.allclose(o.hidden_states[-1], o.last_hidden_state, atol=1e-4))

# kiểm tra cây phân rã: 3 xác suất phải không âm và tổng mỗi hàng = 1
with torch.no_grad():
    b = next(iter(val_loader))
    hl, sl = model(b[0][:2].to(device), b[1][:2].to(device))
    p = compound_probs(hl, sl)
    print('tổng mỗi hàng (phải ≈1):', p.sum(dim=-1).detach().cpu().numpy())



## Train (2-phase or single-phase)


In [ ]:
if TWO_PHASE:
    # PHA 1: mmBERT FROZEN hẳn
    freeze_mmbert(0)
    steps_per_epoch = len(train_loader)
    total_steps_p1 = steps_per_epoch * FREEZE_PHASE_EPOCHS
    optimizer, scheduler = build_optimizer_and_scheduler(
        model, head_lr=PHASE1_HEAD_LR, backbone_lr=0.0,
        num_training_steps=total_steps_p1)
    print(f'--- PHA 1/2 (frozen): tối đa {FREEZE_PHASE_EPOCHS} epoch, head lr {PHASE1_HEAD_LR} ---')
    train_model(model, tag=final_tag, optimizer=optimizer, scheduler=scheduler,
                num_epochs=FREEZE_PHASE_EPOCHS, patience=5)

    # Nạp bản tốt nhất theo F1 của pha 1 để làm điểm khởi đầu cho pha 2
    model.load_state_dict(torch.load(f'{final_tag}_bestf1.pt', map_location=device))

    # PHA 2: mở N tầng cuối
    freeze_mmbert(MMBERT_UNFREEZE_LAYERS)
    total_steps_p2 = steps_per_epoch * UNFREEZE_PHASE_EPOCHS
    optimizer, scheduler = build_optimizer_and_scheduler(
        model, head_lr=PHASE2_HEAD_LR, backbone_lr=PHASE2_BACKBONE_LR,
        num_training_steps=total_steps_p2)
    final_tag = final_tag + '_ft'
    print(f'--- PHA 2/2 (unfreeze {MMBERT_UNFREEZE_LAYERS} tầng cuối): head lr {PHASE2_HEAD_LR}, '
          f'backbone lr {PHASE2_BACKBONE_LR} ---')
    train_model(model, tag=final_tag, optimizer=optimizer, scheduler=scheduler,
                num_epochs=UNFREEZE_PHASE_EPOCHS, patience=5)
else:
    freeze_mmbert(MMBERT_UNFREEZE_LAYERS)
    total_steps = len(train_loader) * 60
    optimizer, scheduler = build_optimizer_and_scheduler(
        model, head_lr=PHASE2_HEAD_LR, backbone_lr=PHASE2_BACKBONE_LR,
        num_training_steps=total_steps)
    train_model(model, tag=final_tag, optimizer=optimizer, scheduler=scheduler)

print(f"Đã huấn luyện xong -> {final_tag}_bestf1.pt (theo F1) / {final_tag}_bestloss.pt (theo val_loss)")


## Final evaluation & per-language breakdown


In [ ]:

model.load_state_dict(torch.load(f'{final_tag}_bestf1.pt', map_location=device))
print("--- VALIDATION (overall, model tốt nhất theo macro-F1) ---")
metrics, hs_pred, hs_prob, p_hate = evaluate(model, df_val, val_loader)
print_metrics(metrics, 'validation')

print("\nConfusion matrix (hàng = true, cột = pred):")
cm = confusion_matrix(df_val['hs_y'].values, hs_pred, labels=[0, 1, 2])
print(f"{'':<16}" + ''.join(f'{l:>15}' for l in HS_LABELS))
for i, lab in enumerate(HS_LABELS):
    print(f"{lab:<16}" + ''.join(f'{v:>15}' for v in cm[i]))

for lang in df_val['lang'].unique():
    mask = (df_val['lang'] == lang).values          # boolean array, khớp thứ tự với hs_pred
    y_lang = df_val['hs_y'].values[mask]
    lang_metrics = {
        'hs_acc': accuracy_score(y_lang, hs_pred[mask]),
        'hs_f1_macro': f1_score(y_lang, hs_pred[mask], average='macro', zero_division=0),
        'f1_implicit': f1_score(y_lang, hs_pred[mask], labels=[1], average='macro', zero_division=0),
        'f1_explicit': f1_score(y_lang, hs_pred[mask], labels=[2], average='macro', zero_division=0),
    }
    print(f"  [{lang}] acc={lang_metrics['hs_acc']:.3f} | macro_f1={lang_metrics['hs_f1_macro']:.3f} "
          f"| f1_implicit={lang_metrics['f1_implicit']:.3f} | f1_explicit={lang_metrics['f1_explicit']:.3f}")

print("\n--- Head 1 đánh giá riêng như bài nhị phân no/yes ---")
hate_true = df_val['hate_bin'].values
hate_pred = (p_hate >= 0.5).astype(int)
print(f"  hate_acc = {accuracy_score(hate_true, hate_pred):.4f}")
print(f"  hate_f1  = {f1_score(hate_true, hate_pred, average='macro', zero_division=0):.4f}")

print("\n--- Head 2 đánh giá riêng trên tập mẫu có hate ---")
h_mask = hate_true == 1
if h_mask.any():
    sub_true = df_val['subtype_bin'].values[h_mask]          # 1 = implicit, 0 = explicit
    sub_pred = (hs_prob[h_mask, 1] >= hs_prob[h_mask, 2]).astype(int)  # P(Imp) >= P(Exp)
    print(f"  n = {h_mask.sum()}")
    print(f"  subtype_acc = {accuracy_score(sub_true, sub_pred):.4f}")
    print(f"  subtype_f1  = {f1_score(sub_true, sub_pred, average='macro', zero_division=0):.4f}")



## Export config (Task B)


In [ ]:

# import pickle

# cfg = {
#     'TWO_PHASE': TWO_PHASE,
#     'FREEZE_PHASE_EPOCHS': FREEZE_PHASE_EPOCHS,
#     'UNFREEZE_PHASE_EPOCHS': UNFREEZE_PHASE_EPOCHS,
#     'MMBERT_UNFREEZE_LAYERS': MMBERT_UNFREEZE_LAYERS,
#     'MMBERT_MODEL_NAME': MMBERT_MODEL_NAME,
#     'MMBERT_DIM': MMBERT_DIM,
#     'MAX_LEN': MAX_LEN,
#     'HS_LABELS': HS_LABELS,
#     'SUBTYPE_WEIGHT': SUBTYPE_WEIGHT,
# }
# with open('stereoqueer_config_taskB.pkl', 'wb') as f:
#     pickle.dump(cfg, f)
# print('Đã lưu: stereoqueer_config_taskB.pkl')



## Export predictions (Task B)


In [ ]:

# # Xuất đúng định dạng SemEval Task B: cột nhãn dự đoán phẳng 3 lớp
# results = df_val[['StereoQueerEval_id', 'lang', 'yt_comment', 'hate_speech']].copy()
# results['prediction'] = [HS_LABELS[i] for i in hs_pred]      # no / yes_implicit / yes_explicit
# for i, lab in enumerate(HS_LABELS):
#     results[f'prob_{lab}'] = hs_prob[:, i]
# results['p_hate'] = p_hate                                    # xác suất yes/no của head 1
# results.to_csv('stereoquereval_val_predictions_taskB.csv', index=False)

# pd.set_option('display.max_colwidth', 60)
# print(results.head(10).to_string())
